# Hurricane Helene (2024): AI Model Track and Intensity Verification

Three AI weather models, Google DeepMind's **GraphCast**, NVIDIA's **SFNO** and Microsoft's **Aurora**, forecast Hurricane Helene 72 h before its `2024-09-27 00:00 UTC` landfall on the Florida Big Bend. All three start at `2024-09-24 00:00 UTC` from the same ERA5 fields (read through the ARCO data source) and step forward every 6 h. In every forecast field the storm center is the maximum of the smoothed 850 hPa relative vorticity and the intensity is the minimum mean sea level pressure (MSLP). ERA5 is the reference: the same two quantities are measured on ERA5 fields, and each model's position error (km) and MSLP error (hPa) are scored against them. You will see how to run three models one after another on a single GPU, how to turn gridded fields into a cyclone track, and how the errors grow with forecast lead time.

> **Measured on an NVIDIA L40:** _filled in after the verified run._

1. Setup and configuration
2. GraphCast forecasts
3. Aurora forecasts
4. SFNO forecasts
5. ERA5 truth track
6. Load saved fields
7. Track and intensity diagnostics
8. Skill tables vs ERA5
9. Figures
10. Summary


## 1 · Setup and configuration

Everything a reader may want to change (the event, initial time, domains and tracker settings) is set in one config cell, and the helper functions below it only read those constants.


In [ ]:
# Environment: must be set before torch / earth2studio are imported.
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"  # GraphCast runs on JAX: allocate on demand so it shares the GPU with torch
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"  # and hand freed memory back to the driver, so Aurora and SFNO can use it after GraphCast


In [ ]:
import gc
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import torch
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from tqdm import tqdm
from scipy.ndimage import gaussian_filter

from earth2studio.models.px import GraphCastOperational, SFNO, Aurora
from earth2studio.data import ARCO, prep_data_array
import earth2studio.run as run
from earth2studio.io import ZarrBackend

from metrics import haversine_km
from tc_skill import lead_error_table, track_skill_table

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


def free_gpu(*names: str) -> None:
    """Delete the named notebook variables (models, tensors), collect garbage, and release cached CUDA memory."""
    scope = globals()
    for name in names:
        scope.pop(name, None)
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# --- Event and forecast timing ---
STORM_NAME = "Helene"

VERIFICATION_TIME = datetime(2024, 9, 27, 0)   # landfall = verification (valid) time

LEAD_TIMES_H = [72]    # 72h only for now (168/240 retired; list kept parameterized)
LEAD_H = LEAD_TIMES_H[0]

INIT_TIMES = {h: VERIFICATION_TIME - timedelta(hours=h) for h in LEAD_TIMES_H}
N_STEPS    = {h: h // 6 for h in LEAD_TIMES_H}

VALID_TIMES = [INIT_TIMES[LEAD_H] + timedelta(hours=6 * k) for k in range(N_STEPS[LEAD_H] + 1)]

# --- Output folders ---
PROJECT = Path.cwd()  # JupyterLab and nbconvert both start in the notebook's folder
RESULTS_DIR = PROJECT / "docs" / "results"  # small CSV metric tables that the README embeds
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR = Path("./outputs/helene")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Domains ---
# (lon_min, lon_max, lat_min, lat_max), lon in [-180, 180].
DOMAIN = (-100, -65, 15, 40)

# Tighter box than DOMAIN, used only for center-finding/the track map — must exclude other
# systems that can dominate a wider box. Hurricane John sat at the SW corner (16N, 99W) on
# 2024-09-24, so the west edge is trimmed to keep it out.
TRACK_DOMAIN = (-95, -76, 15, 35)

LANDFALL_LAT = 30.0
LANDFALL_LON = -83.7

# --- Models and saved fields ---
MODELS = ["GraphCast", "SFNO", "Aurora"]
FILE_PREFIX = {"GraphCast": "graphcast", "SFNO": "sfno", "Aurora": "aurora"}

FIELD_VARS = ["msl", "u10m", "v10m", "u850", "v850"]

# --- Tracker settings ---
VORT_SMOOTH_SIGMA = 1.5      # gaussian_filter sigma (grid cells) before vorticity argmax
SEARCH_RADIUS_DEG = 5.0      # moving search-box half-width (deg) around the previous center

print(f"{STORM_NAME}: lead={LEAD_H}h  init={INIT_TIMES[LEAD_H]}  steps={N_STEPS[LEAD_H]}  "
      f"valid_times={len(VALID_TIMES)} ({VALID_TIMES[0]:%Y-%m-%d %HZ} → {VALID_TIMES[-1]:%Y-%m-%d %HZ})")


In [ ]:
def extract_field(x, coords, variables, domain):
    """Extract `variables` over `domain` from a model/analysis state (x, coords).

    Returns an xarray.Dataset on (lat, lon), sorted ascending, msl in Pa.
    `domain` = (lon_min, lon_max, lat_min, lat_max) in [-180, 180].
    Used for the ERA5 truth fetch (per valid time).
    """
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    dims = list(coords.keys())
    vax = dims.index("variable")
    var_names = list(coords["variable"])
    lat = np.asarray(coords["lat"], dtype=float)
    lon = np.asarray(coords["lon"], dtype=float) % 360.0

    lon_min, lon_max, lat_min, lat_max = domain
    lon0, lon1 = lon_min % 360.0, lon_max % 360.0
    lat_mask = (lat >= lat_min) & (lat <= lat_max)
    lon_mask = (lon >= min(lon0, lon1)) & (lon <= max(lon0, lon1))

    data_vars = {}
    for v in variables:
        field = np.squeeze(np.take(x, var_names.index(v), axis=vax))
        data_vars[v] = (("lat", "lon"), field[np.ix_(lat_mask, lon_mask)])

    ds = xr.Dataset(data_vars, coords={"lat": lat[lat_mask], "lon": lon[lon_mask]})
    return ds.sortby("lat").sortby("lon")


def domain_output_coords(model, domain, variables):
    """Build `output_coords` for run.deterministic: subselect `variables` + the domain lat/lon box.

    Lat/lon values are taken from the model's own output grid so map_coords matches them exactly.
    `domain` = (lon_min, lon_max, lat_min, lat_max) in [-180, 180]; model lon is in 0–360.
    """
    g = model.output_coords(model.input_coords())
    glat = np.asarray(g["lat"], dtype=float)
    glon = np.asarray(g["lon"], dtype=float)
    lon_min, lon_max, lat_min, lat_max = domain
    lo0, lo1 = lon_min % 360.0, lon_max % 360.0
    box_lat = glat[(glat >= lat_min) & (glat <= lat_max)]
    box_lon = glon[(glon >= min(lo0, lo1)) & (glon <= max(lo0, lo1))]
    return {"variable": np.array(variables), "lat": box_lat, "lon": box_lon}


def run_track_inference(model, data_source, init_time, nsteps, store_path,
                        domain=DOMAIN, variables=FIELD_VARS):
    """Integrate `model` for `nsteps` 6h steps from `init_time`, writing EVERY step
    (msl + 10m/850hPa winds over the domain box) to a Zarr store via run.deterministic.

    Skips if the store already exists. run.deterministic fetches the initial conditions itself
    (`lead_time` from the model's input_coords), so GraphCast and Aurora's two-frame init need no
    special handling.
    """
    store_path = Path(store_path)
    if store_path.exists():
        print(f"  cached → {store_path}")
        return
    out_coords = domain_output_coords(model, domain, variables)
    backend = ZarrBackend(str(store_path), backend_kwargs={"overwrite": True})
    run.deterministic([init_time], nsteps, model, data_source, backend,
                      output_coords=out_coords, device=DEVICE)
    print(f"  saved  → {store_path}  (vars={variables}, steps={nsteps + 1})")

In [ ]:
def run_era5_track(data_source, valid_times, save_path, variables=FIELD_VARS):
    """Fetch ERA5 truth fields over DOMAIN at each of `valid_times` and concat on valid_time.

    Reuses extract_field per single time (avoids the time-axis squeeze issue), keeps the
    corrupted-cache-chunk fallback. Cached to a single NetCDF (valid_time, lat, lon).
    """
    if Path(save_path).exists():
        print(f"  cached → {save_path}")
        return xr.open_dataset(save_path)

    snaps = []
    for vt in tqdm(valid_times, desc="ERA5 valid times"):
        try:
            da = data_source(vt, list(variables))
        except RuntimeError:
            # Corrupted cache chunk — retry with a fresh uncached download.
            da = ARCO(cache=False)(vt, list(variables))
        x, coords = prep_data_array(da, device=DEVICE)
        snap = extract_field(x, coords, variables, DOMAIN)
        snaps.append(snap.expand_dims(valid_time=[np.datetime64(vt)]))

    ds = xr.concat(snaps, dim="valid_time")
    ds.to_netcdf(save_path)
    print(f"  saved  → {save_path}  ({ds.sizes['valid_time']} valid times)")
    return ds

In [ ]:
# WB2ERA5 has a hard cutoff of 2023-01-11; Helene's init times are in Sep 2024, so use ARCO.
data_source = ARCO(cache=True)                      # ERA5: the initial state of all three models and the reference

## 2 · GraphCast forecasts

GraphCast (Google DeepMind) — a graph neural network that forecasts the global atmosphere on a 0.25° grid at
13 pressure levels, 6 h per step. This is the operational version, pre-trained on ERA5 (1979–2017) and then
fine-tuned on ECMWF's operational HRES data (2016–2021). It needs two initial frames (t−6 h and t) and is
started here from ERA5 (`data_source`), like the other two models. It runs on JAX rather than torch. Google
now calls the model WeatherNext 1-Graph; Earth2Studio keeps the name GraphCast.

In [ ]:
%%time
try:
    graphcast = GraphCastOperational.load_model(GraphCastOperational.load_default_package())
    run_track_inference(
        graphcast, data_source,
        INIT_TIMES[LEAD_H], N_STEPS[LEAD_H],
        OUT_DIR / "graphcast_72h.zarr",
    )
finally:
    free_gpu("graphcast")

## 3 · Aurora forecasts

Aurora (Microsoft) — 0.25° foundation model, 6-hourly autoregressive. Needs two
initial frames (t−6 h and t); initialized from ARCO ERA5 (`data_source`). Each model is
freed from the GPU as soon as its forecast is saved, so only one is resident at a time.


In [ ]:
%%time
try:
    aurora = Aurora.load_model(Aurora.load_default_package())
    run_track_inference(
        aurora, data_source,
        INIT_TIMES[LEAD_H], N_STEPS[LEAD_H],
        OUT_DIR / "aurora_72h.zarr",
    )
finally:
    free_gpu("aurora")


## 4 · SFNO forecasts

SFNO (NVIDIA) — Spherical Fourier Neural Operator using spectral convolutions
on the sphere. Efficient O(N log N) transforms via spherical harmonics. Initialized from
ARCO ERA5 (`data_source`).


In [ ]:
%%time
try:
    sfno = SFNO.load_model(SFNO.load_default_package())
    run_track_inference(
        sfno, data_source,
        INIT_TIMES[LEAD_H], N_STEPS[LEAD_H],
        OUT_DIR / "sfno_72h.zarr",
    )
finally:
    free_gpu("sfno")


## 5 · ERA5 truth track

Fetch ERA5 fields at every 6-hourly valid time across the window — the reference track each model is compared against.


In [ ]:
%%time
era5_track = run_era5_track(
    data_source, VALID_TIMES,
    OUT_DIR / "era5_track_72h.nc",
)

## 6 · Load saved fields

Reload-safe: works whether you just ran inference or restarted the kernel.


In [ ]:
def load_model_track(store_path, init_time):
    """Open a model's saved Zarr store and index it by valid time (init time + lead time)."""
    ds = xr.open_zarr(store_path, consolidated=False).squeeze("time", drop=True)
    vt = np.datetime64(init_time) + ds["lead_time"].values            # init + lead → valid time
    ds = ds.assign_coords(valid_time=("lead_time", vt)).swap_dims({"lead_time": "valid_time"})
    return ds.sortby("lat").sortby("lon")

model_tracks = {
    m: load_model_track(OUT_DIR / f"{FILE_PREFIX[m]}_72h.zarr", INIT_TIMES[LEAD_H])
    for m in MODELS
}
era5_track = xr.open_dataset(OUT_DIR / "era5_track_72h.nc").sortby("lat").sortby("lon")

for m in MODELS:
    ds = model_tracks[m]
    print(f"{m:8s} steps={ds.sizes['valid_time']:2d}  msl{tuple(ds['msl'].shape)}  "
          f"vars={list(ds.data_vars)}")
print(f"ERA5     steps={era5_track.sizes['valid_time']:2d}  msl{tuple(era5_track['msl'].shape)}")

## 7 · Track and intensity diagnostics

Post-processing only (no GPU), over **`TRACK_DOMAIN`**. For each 6-hourly step:
- **position** = smoothed **850 hPa relative-vorticity maximum**, gated to ±`SEARCH_RADIUS_DEG` around the previous center;
- **intensity** = **minimum MSLP** (hPa) in that gated box.

The helpers below extract the tracks; the plotting functions used in section 9 are defined here too.


In [ ]:
R_EARTH_M = 6.371e6


def crop_to(ds, domain):
    """Crop a (…, lat, lon) Dataset to `domain`=(lon_min,lon_max,lat_min,lat_max) in [-180,180].
    Data lon is 0–360 ascending, so convert the box edges with % 360 before slicing."""
    lon_min, lon_max, lat_min, lat_max = domain
    return ds.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min % 360.0, lon_max % 360.0))


def relative_vorticity(u, v, lat, lon):
    """850-hPa relative vorticity ζ = ∂v/∂x − ∂u/∂y on a regular lat/lon grid (s⁻¹).

    u, v : (nlat, nlon) arrays on ascending lat/lon (m/s). lat, lon : 1-D degrees, ascending.
    Spherical metric terms: dx = R·cosφ·dλ, dy = R·dφ. NH cyclonic ζ > 0.
    """
    lat_r = np.radians(np.asarray(lat, float))
    lon_r = np.radians(np.asarray(lon, float))
    dphi = np.gradient(lat_r)
    dlam = np.gradient(lon_r)
    dy = R_EARTH_M * dphi
    dx = R_EARTH_M * np.cos(lat_r)[:, None] * dlam[None, :]
    dv_dx = np.gradient(v, axis=1) / dx
    du_dy = np.gradient(u, axis=0) / dy[:, None]
    return dv_dx - du_dy


def storm_center_intensity(ds_step, prev_center=None,
                           search_radius_deg=SEARCH_RADIUS_DEG, smooth_sigma=VORT_SMOOTH_SIGMA):
    """Locate the TC center (smoothed 850-hPa vorticity max) and intensity (min MSLP) for one step.

    prev_center=(lat,lon) gates BOTH the vorticity-max and MSLP-min searches to a ±search_radius_deg
    box around it (continuity); None searches the whole (already-cropped) box — used at step 0.
    Returns center_lat/lon, min_mslp_hpa, mslp_lat/lon (separation cross-check), on_edge.
    """
    lat = ds_step["lat"].values; lon = ds_step["lon"].values
    u = ds_step["u850"].values; v = ds_step["v850"].values
    mslp = ds_step["msl"].values / 100.0                     # Pa → hPa

    zeta = relative_vorticity(u, v, lat, lon)
    if smooth_sigma:
        zeta = gaussian_filter(zeta, sigma=smooth_sigma)

    if prev_center is not None:                              # moving search box (continuity gate)
        plat, plon = prev_center
        lat_ok = np.abs(lat - plat) <= search_radius_deg
        dlon = np.abs(((lon - plon + 180.0) % 360.0) - 180.0)
        lon_ok = dlon <= search_radius_deg
        mask = lat_ok[:, None] & lon_ok[None, :]
    else:
        mask = np.ones((lat.size, lon.size), dtype=bool)

    zeta_s = np.where(mask, zeta, -np.inf)                   # gate position (vorticity max) …
    mslp_s = np.where(mask, mslp, np.inf)                    # … and intensity (MSLP min)

    iy, ix = np.unravel_index(np.nanargmax(zeta_s), zeta.shape)
    on_edge = iy in (0, lat.size - 1) or ix in (0, lon.size - 1)
    im, jm = np.unravel_index(np.nanargmin(mslp_s), mslp.shape)
    return {
        "center_lat": float(lat[iy]),
        "center_lon": float(((lon[ix] + 180.0) % 360.0) - 180.0),    # → [-180,180]
        "min_mslp_hpa": float(mslp[im, jm]),
        "mslp_lat": float(lat[im]),
        "mslp_lon": float(((lon[jm] + 180.0) % 360.0) - 180.0),
        "on_edge": bool(on_edge),
    }


def track_from_dataset(ds_ts, seed_center=None):
    """Walk a (valid_time, lat, lon) Dataset forward → per-step center/intensity DataFrame.

    seed_center seeds step 0's search box; None lets step 0 take the global max in the box
    (fine once the box is tight enough to contain only the target storm)."""
    recs = []; prev = seed_center
    for t in ds_ts["valid_time"].values:
        ci = storm_center_intensity(ds_ts.sel(valid_time=t), prev_center=prev)
        ci["valid_time"] = pd.Timestamp(t)
        recs.append(ci)
        prev = (ci["center_lat"], ci["center_lon"])
    return pd.DataFrame(recs)


def build_tracks(model_tracks, era5_track, track_domain=None):
    """ERA5 + each model track over TRACK_DOMAIN (tighter box that excludes other systems).

    ERA5 takes the global vorticity max at step 0; each model is seeded from the ERA5 step-0
    center (disambiguates a weak init step without biasing the track, since the ±gate is wide).
    """
    td = track_domain if track_domain is not None else TRACK_DOMAIN
    era5_df = track_from_dataset(crop_to(era5_track, td), seed_center=None)
    seed = (float(era5_df.iloc[0]["center_lat"]), float(era5_df.iloc[0]["center_lon"]))
    model_dfs = {m: track_from_dataset(crop_to(ds, td), seed_center=seed)
                 for m, ds in model_tracks.items()}
    return era5_df, model_dfs


In [ ]:
def plot_tracks(era5_df, model_dfs, domain, save_path=None):
    """Storm-center tracks (850 hPa vorticity max) — ERA5 vs each model — over the domain."""
    fig = plt.figure(figsize=(8, 7))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor="0.15")
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, edgecolor="0.4")
    ax.set_extent(domain, crs=ccrs.PlateCarree())

    ax.plot(era5_df["center_lon"], era5_df["center_lat"], "-o", color="k", lw=2.5, ms=4,
            label="ERA5", transform=ccrs.PlateCarree(), zorder=6)
    for m, df in model_dfs.items():
        ax.plot(df["center_lon"], df["center_lat"], "-o", lw=1.5, ms=3,
                label=m, transform=ccrs.PlateCarree(), zorder=5)
    ax.plot(LANDFALL_LON, LANDFALL_LAT, marker="*", color="red", ms=16,
            markeredgecolor="white", markeredgewidth=0.8,
            transform=ccrs.PlateCarree(), zorder=10)

    gl = ax.gridlines(draw_labels=True, linewidth=0.2, color="0.7")
    gl.top_labels = gl.right_labels = False
    ax.legend(loc="best", fontsize=9)
    ax.set_title(f"{STORM_NAME} — {LEAD_H} h forecast tracks (850 hPa vorticity centers) vs ERA5")
    if save_path:
        fig.savefig(str(save_path), dpi=150, bbox_inches="tight"); print(f"saved → {save_path}")
    plt.show()


def plot_intensity(era5_df, model_dfs, save_path=None):
    """Intensity (min MSLP) vs lead time — ERA5 vs each model."""
    fig, ax = plt.subplots(figsize=(8, 4.5))
    lead = np.arange(len(era5_df)) * 6
    ax.plot(lead, era5_df["min_mslp_hpa"], "-o", color="k", lw=2.5, label="ERA5")
    for m, df in model_dfs.items():
        ax.plot(lead, df["min_mslp_hpa"], "-o", lw=1.5, label=m)
    ax.set_xlabel("Lead time (h)"); ax.set_ylabel("Min MSLP (hPa)")
    ax.set_xticks(lead)
    ax.invert_yaxis(); ax.grid(alpha=0.3); ax.legend(fontsize=9)
    ax.set_title(f"{STORM_NAME} — Minimum MSLP vs Lead Time")
    if save_path:
        fig.savefig(str(save_path), dpi=150, bbox_inches="tight"); print(f"saved → {save_path}")
    plt.show()


def plot_position_error(era5_df, model_dfs, save_path=None):
    """Track position error (haversine km) vs lead time — each model vs ERA5."""
    fig, ax = plt.subplots(figsize=(8, 4.5))
    lead = np.arange(len(era5_df)) * 6
    for m, df in model_dfs.items():
        err = haversine_km(df["center_lat"].values, df["center_lon"].values,
                           era5_df["center_lat"].values, era5_df["center_lon"].values)
        ax.plot(lead, err, "-o", lw=1.5, label=m)
    ax.set_xlabel("Lead time (h)"); ax.set_ylabel("Position error (km)")
    ax.set_xticks(lead)
    ax.grid(alpha=0.3); ax.legend(fontsize=9)
    ax.set_title(f"{STORM_NAME} — track position error vs ERA5")
    if save_path:
        fig.savefig(str(save_path), dpi=150, bbox_inches="tight"); print(f"saved → {save_path}")
    plt.show()

## 8 · Skill tables vs ERA5

Each model's track is scored against the ERA5 track at every 6-hourly step. Position error is the great-circle distance between the model's storm center and ERA5's; the MSLP error is the model's minimum MSLP minus ERA5's, so a positive value means the model's storm is weaker (higher pressure) than ERA5's. The first table summarizes the whole forecast per model, the second lists the errors at every lead hour; both are saved as CSV files in `docs/results/`.

> **Truth caveat.** ERA5 is a 0.25° reanalysis that smooths storm cores, so the intensity errors below measure agreement with ERA5, not with best-track central pressure.


In [ ]:
era5_df, model_dfs = build_tracks(model_tracks, era5_track)

df_track = track_skill_table(era5_df, model_dfs)
print(f"\n=== {STORM_NAME}: {LEAD_H} h track skill vs ERA5 "
      f"(vorticity center, min-MSLP intensity) ===\n")
print(df_track.to_string(index=False))
df_track.to_csv(RESULTS_DIR / "helene_forecast_comparison_metrics.csv", index=False)

for label, df in {"ERA5": era5_df, **model_dfs}.items():
    n_edge = int(df["on_edge"].sum())
    if n_edge:
        print(f"{label}: {n_edge} step(s) with center on domain edge")

In [ ]:
lead_df = lead_error_table(era5_df, model_dfs, INIT_TIMES[LEAD_H])
lead_df.to_csv(RESULTS_DIR / "helene_forecast_comparison_lead_errors.csv", index=False)
print(f"=== {STORM_NAME}: error by forecast lead hour, vs ERA5 ===\n")
print(lead_df.to_string(index=False))


## 9 · Figures

Three figures show the same comparison in map view, as pressure and as distance: where each storm went, how strong it was, and how far its center was from ERA5's.


In [ ]:
plot_tracks(era5_df, model_dfs, TRACK_DOMAIN, save_path=OUT_DIR / "track_map_72h.png")

**How to read this.** The black line is the ERA5 storm-center track and each colored line is one model's forecast track, with a dot every 6 h starting at the initial time. The red star marks the approximate landfall location. Look for how closely each colored line follows the black one, whether a model runs ahead of or behind ERA5 along the track (dots farther apart or closer together than the black ones), and whether it turns toward the coast too early or too late.


In [ ]:
plot_intensity(era5_df, model_dfs, save_path=OUT_DIR / "intensity_72h.png")


**How to read this.** The horizontal axis is forecast lead time in hours since initialization; the vertical axis is the minimum MSLP in hPa and is inverted, so a stronger storm (lower pressure) plots higher. A colored line above the black ERA5 line means the model's storm is stronger than ERA5's, below means weaker. Look at whether each model deepens the storm over time and how its curve departs from ERA5's as the lead time grows. ERA5 smooths storm cores, so this compares each model with ERA5, not with the observed central pressure.


In [ ]:
plot_position_error(era5_df, model_dfs, save_path=OUT_DIR / "position_error_72h.png")


**How to read this.** The horizontal axis is forecast lead time in hours; the vertical axis is the great-circle distance in km between a model's storm center and the ERA5 storm center at the same valid time, so zero would be a perfect match with ERA5. Look at how quickly each curve grows with lead time and whether the growth is steady or jumps at a single step, which can mean the vorticity maximum moved to a different feature.


## 10 · Summary

- **Track map** (`outputs/helene/track_map_72h.png`): does each model's forecast storm path stay on the ERA5 path toward the Florida Big Bend, and is it early or late along it?
- **Skill table** (`docs/results/helene_forecast_comparison_metrics.csv`): how large is each model's mean and maximum position error, and is its pressure biased weak or strong relative to ERA5?
- **Lead-hour table** (`docs/results/helene_forecast_comparison_lead_errors.csv`): at which forecast hours do the position and MSLP errors start to grow?
- **Intensity series** (`outputs/helene/intensity_72h.png`): does each model deepen the storm the way ERA5 does, and does it stay above or below ERA5's pressure curve?
- **Position error vs lead time** (`outputs/helene/position_error_72h.png`): how fast does each model's track drift away from ERA5's?

The numbers live in the CSV files in `docs/results/` and change with every run. `helene_forecast_comparison_wuduan.ipynb` repeats the SFNO and Aurora comparison with the Wu–Duan cyclone tracker instead of the vorticity center.
